#### Sunwoo Lee - Z23597551
#### Dr Dingding Wang
#### CAP 6640 - Natural Language Processing
#### 29th March 2026

# Assignment 3 - Sentiment Analysis using Textblob and Vader

## Imports and Setup

In [20]:
!pip install textblob nltk vadersentiment

Defaulting to user installation because normal site-packages is not writeable


In [29]:
import os
import pandas as pd
import nltk
from sklearn.metrics import confusion_matrix, classification_report
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [42]:
analyzer = SentimentIntensityAnalyzer()

# Dataset was pre unzipped using 7-zip
DATA_PATH = 'txt_sentoken'
NEG_PATH = os.path.join(DATA_PATH, "neg")
POS_PATH = os.path.join(DATA_PATH, "pos")

## Loading Dataset Into DF

In [17]:
texts = []
labels = []

# Load negative reviews
for file in os.listdir(NEG_PATH):
    file_path = os.path.join(NEG_PATH, file)
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        texts.append(f.read())
        labels.append("neg")

# Load positive reviews
for file in os.listdir(POS_PATH):
    file_path = os.path.join(POS_PATH, file)
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        texts.append(f.read())
        labels.append("pos")

df = pd.DataFrame({
        "text": texts,
        "label": labels
    })

print(df.head())
print(df["label"].value_counts())

                                                text label
0  plot : two teen couples go to a church party ,...   neg
1  the happy bastard's quick movie review \ndamn ...   neg
2  it is movies like these that make a jaded movi...   neg
3   " quest for camelot " is warner bros . ' firs...   neg
4  synopsis : a mentally unstable man undergoing ...   neg
label
neg    1000
pos    1000
Name: count, dtype: int64


## Small Preprocessing

In [64]:
# Using code from https://www.youtube.com/watch?v=AnvrJNLKp0k
# Lower casing and removing punctuations

df["text"] = df['text'].apply(lambda x: " ".join(x.lower() for x in x.split()))
df["text"] = df['text'].str.strip().replace(r'[^\w\s]', "", regex=True)

In [66]:
df.text.head(5)

0    plot two teen couples go to a church party dri...
1    the happy bastards quick movie review damn tha...
2    it is movies like these that make a jaded movi...
3    quest for camelot is warner bros first feature...
4    synopsis a mentally unstable man undergoing ps...
Name: text, dtype: object

## Textblob Sentiment Analysis

In [68]:
def get_textblob_polarity(text):
    return TextBlob(text).sentiment.polarity

def textblob_label(score):
    return "pos" if score >= 0 else "neg"

df["textblob_polarity"] = df["text"].apply(get_textblob_polarity)
df["textblob_pred"] = df["textblob_polarity"].apply(textblob_label)

### Result

In [91]:
tb_accuracy = (df["textblob_pred"] == df["label"]).mean()
print(f"TextBlob Accuracy: {tb_accuracy*100}%")

print("\nConfusion Matrix:")
tb_cm = confusion_matrix(df["label"], df["textblob_pred"])
print(pd.DataFrame(
    tb_cm,
    index=["Actual Neg", "Actual Pos"],
    columns=["Predicted Neg", "Predicted Pos"]
))
print("\nClassification Report:")
print(classification_report(df["label"], df["textblob_pred"]))

TextBlob Accuracy: 59.8%

Confusion Matrix:
            Predicted Neg  Predicted Pos
Actual Neg            225            775
Actual Pos             29            971

Classification Report:
              precision    recall  f1-score   support

         neg       0.89      0.23      0.36      1000
         pos       0.56      0.97      0.71      1000

    accuracy                           0.60      2000
   macro avg       0.72      0.60      0.53      2000
weighted avg       0.72      0.60      0.53      2000



## VADER Sentiment Analysis

In [75]:
def get_vader_compound(text):
    return analyzer.polarity_scores(text)["compound"]

def vader_label(score):
    return "pos" if score >= 0 else "neg"

df["vader_compound"] = df["text"].apply(get_vader_compound)
df["vader_pred"] = df["vader_compound"].apply(vader_label)

### Result 

In [93]:
v_accuracy = (df["vader_pred"] == df["label"]).mean()
print(f"VADER Accuracy: {v_accuracy*100}%")

print("\nConfusion Matrix:")
v_cm = confusion_matrix(df["label"], df["vader_pred"])
print(pd.DataFrame(
    v_cm,
    index=["Actual Neg", "Actual Pos"],
    columns=["Predicted Neg", "Predicted Pos"]
))
print("\nClassification Report:")
print(classification_report(df["label"], df["vader_pred"]))

VADER Accuracy: 64.35%

Confusion Matrix:
            Predicted Neg  Predicted Pos
Actual Neg            444            556
Actual Pos            157            843

Classification Report:
              precision    recall  f1-score   support

         neg       0.74      0.44      0.55      1000
         pos       0.60      0.84      0.70      1000

    accuracy                           0.64      2000
   macro avg       0.67      0.64      0.63      2000
weighted avg       0.67      0.64      0.63      2000

